# MUnitQuest – Algorithm Challenge Tutorial
# Showdown Phase: Dynamic Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUnitQuest/MUnitQuest_tutorials/blob/main/algorithm_challenge_tutorials/04_showdown_dynamic.ipynb)


This notebook walks through the full workflow for the **Showdown Phase** of the dynamic task:

1. **Download** the showdown dataset
2. **Explore** a recording – file structure, effort and joint-angle profile, and the scored segments
3. **Decompose** the HD-EMG with CBSS (the reference algorithm bundled with `muniverse`)
4. **Export** your spike trains and process metadata, ready for the Codabench upload

Good luck on your quest 💪

---

#### Set up the environment

```bash
python3.11 -m venv .venv && source .venv/bin/activate
pip install muniverse-emg
```

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    os.system("pip install muniverse-emg")
    os.system("git clone --depth 1 https://github.com/MUnitQuest/MUnitQuest_tutorials.git")
    sys.path.insert(0, "MUnitQuest_tutorials/algorithm_challenge_tutorials")

In [ ]:
import json
from importlib.resources import files
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

from muniverse.algorithms.decomposition import decompose_cbss
from muniverse.utils.bids_routines import read_edf

## 1 – Download the dataset

The showdown dataset is hosted on DaRUS, the data repository of the University of Stuttgart, in the [MUnitQuest dataverse](https://darus.uni-stuttgart.de/dataverse/munitquest-algorithm-challenge). It contains **50 dynamic recordings** from a variety of simulators.

Each recording is stored in **BIDS-flat** structure: a folder per recording with an EDF file containing the HD-EMG signals, the force target and the joint angle, accompanied by sidecar files:

- `*_emg.edf` — HD-EMG signal (all channels) + `Force` + `JointAngle`
- `*_channels.tsv` — channel names, types, units and `status`
- `*_electrodes.tsv` — electrode positions (x, y in mm)
- `*_emg.json` — recording metadata (sampling frequency, reference, …)
- `*_events.tsv` — movement and effort phases, including the **scored segments**

Compared to the familiarisation set, note that:

- A recording has **one or several scored segments** (`scoring_start` / `scoring_end`), one for every movement segment. Effort ramps, holds and rest are not scored.
- `Force` is the force target in % MVC and `JointAngle` is the wrist angle in degrees (sign convention in the channel description).

In [ ]:
DATASET_ROOT = Path("../data/Showdown-Dynamic")

In [ ]:
# Download the "dynamic" folder of the showdown dataset from DaRUS and unpack it so that it contains
# one folder per recording, e.g. DATASET_ROOT / "recording-001_challenge-dynamic".
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"{DATASET_ROOT} not found. Download the showdown dynamic data from the MUnitQuest dataverse on DaRUS "
        "(https://darus.uni-stuttgart.de/dataverse/munitquest-algorithm-challenge) and unpack it there."
    )
print(f"{len(list(DATASET_ROOT.glob('recording-*')))} recordings in {DATASET_ROOT}")

In [ ]:
def load_recording(edf_path):
    """Load a showdown recording (EMG-BIDS, flat layout) and its sidecar files.

    Parameters
    ----------
    edf_path : str or Path
        Path to the *_emg.edf file.

    Returns
    -------
    dict with keys:
        emg        - (n_emg_channels, n_samples) HD-EMG in mV, all grids, bad channels included
        good       - (n_emg_channels,) bool, False for channels marked bad in *_channels.tsv
        force      - (n_force, n_samples) force / effort channel(s)
        force_unit - list of str, unit of each force channel ("percent MVC" or "n/a")
        angle      - (n_samples,) joint angle in degrees, or None if the recording has no angle channel
        fsamp      - int, sampling frequency in Hz (differs between recordings)
        channels   - DataFrame from *_channels.tsv (EMG channels only), with electrode x / y in mm
        events     - DataFrame from *_events.tsv
        task_name  - str, short task description
        edf_path   - Path object
    """
    edf_path = Path(edf_path)
    stem = edf_path.stem  # e.g. recording-004_challenge-dynamic_emg

    # Read EDF: signals shape is (n_channels, n_samples)
    signals, sig_hdrs, hdr = read_edf(str(edf_path))

    # Sidecar files
    channels_df   = pd.read_csv(edf_path.with_name(stem.replace("_emg", "_channels") + ".tsv"), sep="\t")
    electrodes_df = pd.read_csv(edf_path.with_name(stem.replace("_emg", "_electrodes") + ".tsv"), sep="\t")
    events_df     = pd.read_csv(edf_path.with_name(stem.replace("_emg", "_events") + ".tsv"), sep="\t")
    with open(edf_path.with_suffix(".json")) as f:
        meta = json.load(f)
    fsamp = int(meta["SamplingFrequency"])

    # *_channels.tsv lists the channels in the order of the EDF signals
    assert signals.shape[0] == len(channels_df), "EDF and channels.tsv disagree"

    emg_mask   = (channels_df["type"].str.upper() == "EMG").to_numpy()
    force_mask = channels_df["name"].str.lower().str.startswith("force").to_numpy()
    angle_mask = (channels_df["name"].str.lower() == "jointangle").to_numpy()

    # EMG channels: keep the table row of every channel and add the electrode position (mm)
    emg_channels = channels_df[emg_mask].reset_index(drop=True)
    emg_channels = emg_channels.merge(electrodes_df[["name", "x", "y"]].rename(columns={"name": "signal_electrode"}),
                                      on="signal_electrode", how="left")
    emg  = signals[emg_mask]                                   # [mV]
    good = (emg_channels["status"] == "good").to_numpy()

    angle = signals[angle_mask][0] if angle_mask.any() else None

    return {
        "emg":        emg,
        "good":       good,
        "force":      signals[force_mask],
        "force_unit": channels_df.loc[force_mask, "units"].fillna("n/a").tolist(),
        "angle":      angle,
        "fsamp":      fsamp,
        "channels":   emg_channels,
        "events":     events_df,
        "task_name":  meta.get("TaskName", ""),
        "edf_path":   edf_path,
    }

In [ ]:
def scoring_windows(events_df):
    """(start, end) of every scored segment in seconds, from the scoring_start / scoring_end events."""
    starts = events_df.query("event_type == 'scoring_start'")["onset"].to_numpy()
    ends   = events_df.query("event_type == 'scoring_end'")["onset"].to_numpy()
    return list(zip(starts, ends))

In [ ]:
edf_path = DATASET_ROOT / "recording-004_challenge-dynamic/recording-004_challenge-dynamic_emg.edf"

rec = load_recording(edf_path)
print(f"Task:            {rec['task_name']}")
print(f"Electrode array: {rec['emg'].shape[0]} channels ({(~rec['good']).sum()} marked bad)")
print(f"Sampling rate:   {rec['fsamp']} Hz")
print(f"Duration:        {rec['emg'].shape[1] / rec['fsamp']:.1f} s")
print(f"Scored segments: {[(float(round(s, 2)), float(round(e, 2))) for s, e in scoring_windows(rec['events'])]} s")

## 2 – Explore the recording

`load_recording()` parses the EDF and its sidecar files and returns a dict:

| Key | Shape / Type | Description |
|---|---|---|
| `emg` | (n_channels, n_samples) | HD-EMG signal in mV |
| `good` | (n_channels,) | `False` for channels marked bad in `*_channels.tsv` |
| `force` | (1, n_samples) | Force target in % MVC |
| `angle` | (n_samples,) | Joint angle in degrees |
| `fsamp` | int | Sampling frequency |
| `channels` | DataFrame | EMG rows of `*_channels.tsv` plus electrode `x`, `y` (mm) |
| `events` | DataFrame | Movement and effort phases from `*_events.tsv` |
| `task_name` | str | Short task description |

The events table describes the trial and marks the **scored segments**:

| `event_type` | Meaning | Scored |
|---|---|---|
| `rest` | Quiet period before and after the trial | no |
| `muscle_on`, `muscle_off` | Onset and offset of the voluntary contraction | – |
| `linear_isometric_ramp` | Effort ramp at a fixed joint angle (`mvc_rate` = direction) | no |
| `isometric_hold` | Steady effort at a fixed joint angle | no |
| `linear_isotonic_ramp` | Joint rotation at constant effort (`joint_velocity` = direction and speed) | **yes** |
| `isotonic_sinusoid` | Sinusoidal joint rotation at constant effort | **yes** |
| `scoring_start`, `scoring_end` | Mark the beginning and the end of every scored segment | – |

In [ ]:
rec['events']

In [ ]:
emg        = rec["emg"]       # (n_channels, n_samples), all EMG channels
good       = rec["good"]      # False for channels marked bad
force      = rec["force"]     # (n_force, n_samples)
fsamp      = rec["fsamp"]
channels   = rec["channels"]
n_channels, n_samples = emg.shape
duration_s = n_samples / fsamp
t = np.arange(n_samples) / fsamp
windows = scoring_windows(rec["events"])

# Best good channel by RMS (for time-series plot)
rms = np.sqrt(np.mean(emg ** 2, axis=1))
rms[~good] = -np.inf
emg_ch = emg[int(np.argmax(rms))]
rms[~good] = np.nan
# bandpass-filter for display
b, a = butter(2, [10.0, 500.0], fs=fsamp, btype="band")
emg_ch = filtfilt(b, a, emg_ch)
angle = rec["angle"]

### Signal overview

Let's plot the three main signals. The **effort profile** (% MVC) is held constant while the wrist moves, the **joint angle** follows the movement of the trial (slow and fast segments, optionally with holds), and the **best EMG channel** (highest RMS, bandpass-filtered 10–500 Hz) gives a quick read on signal quality. The shaded areas are the **scored segments**; anything you predict outside of them is disregarded.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, force[0])
axes[0].set_ylabel("Effort (% MVC)")
axes[0].set_title(f"{rec['task_name']}")

axes[1].plot(t, angle)
axes[1].set_ylabel("Angle (°)")

axes[2].plot(t, emg_ch, lw=0.5)
axes[2].set_ylabel("EMG (mV)")
axes[2].set_xlabel("Time (s)")

for ax in axes:
    for start, end in windows:
        ax.axvspan(start, end, color="tab:green", alpha=0.12)

fig.tight_layout()
plt.show()

## 3 – Decompose with CBSS

CBSS (Convolutive Blind Source Separation) is a standard fastICA-based algorithm for motor unit decomposition. It is the reference algorithm included with `muniverse`.

### What CBSS expects

* Input: **2D array** shaped `(n_channels, n_samples)`
* Channels marked **bad** are left out here (`emg[good]`).

In [ ]:
# Leave out the channels marked bad
emg_2d = emg[good]   # (n_good_channels, n_samples)
print(f"EMG ready for decomposition: {emg_2d.shape[0]} of {n_channels} channels x {n_samples} samples at {fsamp} Hz")

### Choosing the analysis window

Spikes are only scored inside the `scoring_start` / `scoring_end` segments, which cover the **joint movements**. The effort ramps, holds and rest are not scored. CBSS assumes stationary EMG statistics, so the dynamic task is hard for it; as in the familiarisation notebook we simply run it on the span from the first `scoring_start` to the last `scoring_end` and see what it finds.

In [ ]:
# Load and adjust the CBSS configuration
cbss_cfg = json.loads(files("muniverse").joinpath("configs/cbss.json").read_text())

# Analysis window: from the first scoring_start to the last scoring_end
start_time = windows[0][0]
end_time   = windows[-1][1]

cbss_cfg["preProcessingConfig"][2]["t_start"] = start_time
cbss_cfg["preProcessingConfig"][2]["t_end"]   = end_time
cbss_cfg["algorithmConfig"]["ica_iterations"] = 30
cbss_cfg["algorithmConfig"]["verbose"] = False

In [ ]:
# Run decomposition  (≈ 10 s – 1 min depending on hardware)
print("Running CBSS decomposition …")

results, run_metadata = decompose_cbss(
    emg_2d,
    fsamp,
    algorithm_config=cbss_cfg,
    meta={"filename": str(edf_path), "format": "edf"},
)

In [ ]:
spikes     = results["spikes"]        # DataFrame: onset, duration, sample, unit_id, event_type
silhouette = results["scores"]["sil"] # quality score per detected MU (0–1)

n_detected = results["sources"].shape[0]
mean_sil   = float(np.mean(silhouette)) if silhouette is not None and len(silhouette) > 0 else float("nan")
print(f"\nDetected {n_detected} motor units | mean silhouette = {mean_sil:.3f}")

### Decomposition strategy

#TODO: There may or may not be stationary segments for this challenge! So, let's change the text below.

The movement is non-stationary, so the electrode-to-muscle geometry changes over the trial. A natural approach is a two-phase strategy:

1. **Phase 1 – Initialise on a stationary segment.** Apply a standard algorithm (e.g. CBSS) where the EMG statistics are approximately stationary, e.g. around the effort plateau or during a slow movement. This yields stable initial motor unit filters.

2. **Phase 2 – Track across the movement.** Use the filters from Phase 1 as the starting point for an adaptive decomposition algorithm that tracks the motor units across the fast, non-stationary segments.

In this notebook we only demonstrate a plain CBSS run. The exported spike trains are a valid leaderboard submission and a starting point for Phase 2.

## 4 – Visualise decomposition results

In [ ]:
idx = 0

source = results["sources"][idx]  # (n_timestamps,)
t_src = np.arange(len(source)) / fsamp

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_src, source, lw=0.2)
ax.set_title(f"Source {idx} (silhouette = {silhouette[idx]:.3f})")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (a.u.)")
fig.tight_layout()
plt.show()

In [ ]:
if spikes.empty:
    print("No motor units detected – check your configuration or signal quality.")
else:
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(12, 6), gridspec_kw={"height_ratios": [1, 3]}, sharex=True
    )

    ax_top.plot(t, angle, color="#2196F3", lw=1.5)
    ax_top.set_ylabel("Angle (°)")
    ax_top.set_title(f"CBSS decomposition – {n_detected} MUs detected")

    for row_idx, (unit_id, group) in enumerate(spikes.groupby("unit_id")):
        sp_times = group["onset"].values
        ax_bot.vlines(sp_times, row_idx - 0.4, row_idx + 0.4, lw=0.5, color="#E53935", alpha=0.7)

    for ax in (ax_top, ax_bot):
        for start, end in windows:
            ax.axvspan(start, end, color="tab:green", alpha=0.12)

    ax_bot.set_ylabel("Detected motor unit")
    ax_bot.set_xlabel("Time (s)")
    ax_bot.set_xlim(0, duration_s)
    ax_bot.set_ylim(-1, n_detected)

    fig.tight_layout()
    plt.show()

## 5 – Export for Codabench upload

The MUnitQuest leaderboard expects a **BIDS-style tab-separated file** named after the source recording:

```
recording-XXX_challenge-dynamic_desc-decomposed_events.tsv
```

with the following columns:

| Column | Type | Description |
|---|---|---|
| `onset` | float | Spike time in **seconds** |
| `duration` | int | Always `0` (instantaneous event) |
| `sample` | int | Spike sample index |
| `unit_id` | int | Motor unit index (0-based) |
| `event_type` | str | Annotates `"motor-unit-spike"` |

One row per spike discharge. `export_events_file()` below handles the conversion and naming automatically.

In [ ]:
SUBMISSION_FOLDER = Path("../submissions/submission-showdown-dynamic")
SUBMISSION_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_events_file(spike_times, unit_ids, edf_path, fsamp, output_dir=None):
    """Write spike trains to a BIDS-style *_desc-decomposed_events.tsv.

    Parameters
    ----------
    spike_times : array-like of int
        Spike sample indices.
    unit_ids : array-like of int
        Motor unit label for each spike. Must be the same length as
        ``spike_times``.
    edf_path : str or Path
        Path to the source EDF file (used to derive the output filename).
    fsamp : int
        Sampling frequency in Hz.
    output_dir : str or Path, optional
        Directory to write the TSV. Defaults to the same directory as edf_path.

    Returns
    -------
    Path
        Absolute path of the written TSV file.
    """
    spike_times = np.asarray(spike_times)
    unit_ids    = np.asarray(unit_ids)
    assert len(spike_times) == len(unit_ids), (
        f"spike_times and unit_ids must have the same length "
        f"(got {len(spike_times)} and {len(unit_ids)})"
    )

    df = pd.DataFrame({
        "onset":       np.round(spike_times / fsamp, 6),
        "duration":    0,
        "sample":      spike_times.astype(int),
        "unit_id":     unit_ids.astype(int),
        "event_type": "motor-unit-spike",
    })
    df = df.sort_values("onset").reset_index(drop=True)

    edf_path = Path(edf_path)
    stem = edf_path.stem.replace("_emg", "_desc-decomposed")
    out_dir = Path(output_dir) if output_dir else edf_path.parent
    out_path = out_dir / f"{stem}_events.tsv"
    df.to_csv(out_path, sep="\t", index=False, na_rep="n/a")
    return out_path

In [ ]:
out_path = export_events_file(spikes["sample"].values, spikes["unit_id"].values, edf_path, fsamp,
                              output_dir=SUBMISSION_FOLDER)
print(f"Saved decomposition outputs to {out_path}")

## 6 – Prepare metadata for your submission

Every submission to MUnitQuest must include a metadata file alongside each decomposition events TSV, e.g.,

`recording-XXX_challenge-dynamic_desc-decomposed_log.json`

At minimum it must contain:

| Field | Type | Required sub-keys | Description |
|---|---|---|---|
| `GeneratedBy` | `list[dict]` | `Name`, `Description`, `CodeURL`, `License`| Algorithm name, a short plain-English description, and a URL pointing to the code (GitHub, GitLab, or similar). `Version` is optional. **`CodeURL` is required even if your repository is private** — it is used to verify algorithm identity, not to access the code. |
| `Execution` | `dict` | `Runtime` | Total wall-clock execution time for the decomposition (seconds, as a number) |
| `Environment`| `dict` | — | Hardware used: CPU model, total RAM, and GPU (if applicable) |

This metadata helps contextualise performance differences across submissions.

### Tracking execution time

Wrap your decomposition call with `time.time()` to measure wall-clock duration:

```python
import time
t0 = time.time()
results = my_decomposition_algorithm(...)
runtime_seconds = time.time() - t0
```

If you use `muniverse`'s `decompose_cbss`, timing is already captured in `run_metadata['Execution']['Timing']` — no manual instrumentation needed (see below).

### Recording hardware information

The cells below use `platform`, `subprocess`, and `psutil` to collect CPU, RAM, and GPU details automatically.

In [ ]:
import os, platform, subprocess

cpu_info = {
    "model":   platform.processor() or platform.machine() or "unknown",
    "n_cores": os.cpu_count(),
    "system":  platform.system(),
    "node":    platform.node(),
}

try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9)
except ImportError:
    ram_gb = "unknown"

try:
    gpu = subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True
    ).decode().strip()
except Exception:
    gpu = "none"

Now assemble the full metadata dict. The manual approach below works regardless of which decomposition library you use — just substitute `runtime_seconds` with however you measured it:

In [ ]:
from datetime import datetime

t_start = datetime.fromisoformat(run_metadata['Execution']['Timing']['Start'])
t_end   = datetime.fromisoformat(run_metadata['Execution']['Timing']['End'])

runtime_seconds = (t_end - t_start).total_seconds()

Next, add a small description to the metadata_dict

In [ ]:
description: str = "Tutorial notebook to decompose a recording from the MUnitQuest Algorithm Challenge - Showdown Phase, Dynamic Task."

run_metadata["GeneratedBy"][0]["Description"] = description

In [ ]:
metadata_log = {
    "GeneratedBy": [
        {
            "Name": "MUnitQuest Tutorials",
            "Description": description,
            "CodeURL": "https://github.com/MUnitQuest/MUnitQuest_tutorials/tree/main/algorithm_challenge_tutorials/04_showdown_dynamic.ipynb",
            "Version": "n/a",
            "Container": "n/a"
        }
    ],
    "Execution": {
        "Runtime": runtime_seconds
    },
    "RuntimeEnvironment": {
        "CPU": cpu_info,
        "GPU": gpu,
        "RAM": ram_gb
    }
}

If you decompose with `muniverse`'s `decompose_cbss`, the `run_metadata` dict already contains all of the above in a structured form — hardware, timing, and algorithm provenance are captured automatically, so you can build the log in one line:

In [ ]:
metadata_log = {
    "GeneratedBy": run_metadata['GeneratedBy'],
    "Execution": run_metadata['Execution'],
    "RuntimeEnvironment": run_metadata['RuntimeEnvironment']
}

In [ ]:
def export_metadata(metadata_log, edf_path, output_dir=None):
    """Write a metadata log dict to a *_desc-metadata_log.json file.

    Parameters
    ----------
    metadata_log : dict
        Metadata to serialise (GeneratedBy, Runtime, Environment, …).
    edf_path : str or Path
        Path to the source EDF file (used to derive the output filename).
    output_dir : str or Path, optional
        Directory to write the JSON. Defaults to the same directory as edf_path.

    Returns
    -------
    Path
        Absolute path of the written JSON file.
    """
    edf_path = Path(edf_path)
    stem = edf_path.stem.replace("_emg", "_desc-decomposed_log")
    out_dir = Path(output_dir) if output_dir else edf_path.parent
    out_path = out_dir / f"{stem}.json"
    out_path.write_text(json.dumps(metadata_log, indent=2))
    return out_path

In [ ]:
out_path = export_metadata(metadata_log, edf_path, output_dir=SUBMISSION_FOLDER)
print(f"Saved metadata to {out_path}")

## 7 – Decompose the full dataset and submit

You have now decomposed a single showdown recording end-to-end. The showdown set contains **50 dynamic recordings** — repeat the steps above for each one, but with your algorithm! A minimal loop over the dataset looks like this:

```python
for edf_path in sorted(DATASET_ROOT.glob("recording-*/*_emg.edf")):
    rec = load_recording(edf_path)
    windows = scoring_windows(rec["events"])
    emg_2d = rec["emg"][rec["good"]]

    spike_samples, unit_ids, run_metadata = my_decomposition_algorithm(emg_2d, rec["fsamp"], windows)

    export_events_file(spike_samples, unit_ids, edf_path, rec["fsamp"], output_dir=SUBMISSION_FOLDER)
    export_metadata(metadata_log, edf_path, output_dir=SUBMISSION_FOLDER)
```

### Submission format

For each recording, `export_events_file` and `export_metadata` write a pair of files into your results directory:

| File | Content |
|---|---|
| `*_desc-decomposed_events.tsv` | Spike trains (one row per discharge) |
| `*_desc-decomposed_log.json` | Algorithm provenance, runtime, and hardware |

A few things to keep in mind for the showdown phase:

- The `*_events.tsv` sidecar of every recording contains one `scoring_start` / `scoring_end` pair per movement segment (1 to 4 per recording). Only spikes inside these segments are scored, anything outside is disregarded.
- The reference spike trains are hidden. A recording for which you submit no file scores zero, so submit a file for every recording.
- The number of columns (6 or 12) differs between recordings. Read the electrode table instead of hard-coding the array shape.
- To be eligible for awards, your code needs to be openly available (see the [Algorithm Challenge page](https://munitquest.github.io/algorithm-challenge/)). `CodeURL` in the metadata log must point to it.
- Both export functions derive the output filename automatically from the EDF path, so you can loop over recordings without name collisions.

### Validate your submission

Run `validate_submission()` on your results directory to catch formatting errors before upload. Here we point `data_dir` at the single recording that was decomposed above; for a full submission point it at `DATASET_ROOT`.

In [ ]:
from utils import validate_submission

is_valid, errors, warnings = validate_submission(
    submission_dir=SUBMISSION_FOLDER,
    data_dir=DATASET_ROOT / "recording-004_challenge-dynamic",
)

Fix any reported errors before proceeding — warnings (e.g. skipped recordings) are non-fatal. And finally, create a `.zip` archive to submit. Assuming you collected all your decomposition results into `submission-showdown-dynamic/`, simply do

```bash
zip -j my_submission.zip submission-showdown-dynamic/*
```

Then head to the [MUnitQuest Codabench submission page](https://www.codabench.org/competitions/15749/) to compete!